# Wafer Yield Intelligence - Notebook 01: Data foundation
**Jira cards:** WYI-8 (data dictionary), WYI-9 (data-quality log), WYI-10 (yield KPI), WYI-11 (clean table)

Before running: attach this notebook to the Lakehouse `Semi_conductor_raw_data` (left panel, *Add lakehouses*).
Raw table = **bronze** (never edited). Clean table = **silver** (`wafer_clean`).

In [ ]:
# Cell 1 - Load raw table and verify it
import pandas as pd, numpy as np
SCHEMA = "dbo"
RAW = f"{SCHEMA}.semiconductor_yield_forecasting_data"

df = spark.read.table(RAW).toPandas()
print("Rows x columns:", df.shape)          # expect (1250, 28)
print(df.dtypes.to_string())

def save_table(pdf, name):
    spark.createDataFrame(pdf).write.mode("overwrite").format("delta").saveAsTable(f"{SCHEMA}.{name}")
    print("saved ->", f"{SCHEMA}.{name}", len(pdf), "rows")

## WYI-8: Data dictionary

In [ ]:
# Cell 2 - Data dictionary (units are NOT in the file -> question for the client)
rows = [
 ("lot_id","Identifier","Batch of wafers processed together","Start","Yes"),
 ("wafer_id","Identifier","Wafer position/number within the lot","Start","Yes"),
 ("product_type","Context","Product family (CPU, GPU, FPGA, ASIC, Memory)","Start","Yes"),
 ("technology_node","Context","Manufacturing generation (7nm to 28nm)","Start","Yes"),
 ("process_date","Context","Date the lot was processed","Start","Yes"),
 ("etch_tool","Tool","Etch machine used","Etch","Yes"),
 ("litho_tool","Tool","Lithography machine used","Lithography","Yes"),
 ("deposition_tool","Tool","Deposition machine used","Deposition","Yes"),
 ("implant_tool","Tool","Implant machine used","Implant","Yes"),
 ("etch_rate","Process setting","Speed of material removal during etch","Etch","Yes"),
 ("pressure","Process setting","Chamber pressure","Etch","Yes"),
 ("temperature","Process setting","Process temperature","Etch","Yes"),
 ("exposure_time","Process setting","Light exposure duration","Lithography","Yes"),
 ("focus_offset","Process setting","Lithography focus deviation","Lithography","Yes"),
 ("dose","Process setting","Exposure/implant dose","Lithography","Yes"),
 ("deposition_rate","Process setting","Speed of layer deposition","Deposition","Yes"),
 ("thickness_uniformity","Inline metrology","How even the deposited layer is","Deposition","Check with client"),
 ("implant_energy","Process setting","Ion implant energy","Implant","Yes"),
 ("tilt_angle","Process setting","Implant tilt angle","Implant","Yes"),
 ("critical_dimension","Inline metrology","Width of the smallest printed feature","After litho/etch","Check with client"),
 ("oxide_thickness","Inline metrology","Thickness of the oxide layer","After deposition","Check with client"),
 ("resistivity","Inline metrology","Electrical resistivity of material","After implant","Check with client"),
 ("defect_count","Defects","Number of defects found by inspection","Inspection","No"),
 ("defect_density","Defects","Defects per unit area","Inspection","No"),
 ("vth","Electrical test","Transistor threshold voltage","Electrical test","No"),
 ("leakage_current","Electrical test","Current leaking when transistor is off","Electrical test","No"),
 ("resistance","Electrical test","Measured electrical resistance","Electrical test","No"),
 ("yield","TARGET","Share of good dies on the wafer (0 to 1)","Final test","No"),
]
dd = pd.DataFrame(rows, columns=["column","group","meaning","process_stage","available_for_early_warning"])
dd["unit"] = "Unknown - ask client"
dd["data_type"] = dd["column"].map(lambda c: str(df[c].dtype) if c in df.columns else "missing!")
missing = set(df.columns) - set(dd["column"])
print("Columns not documented:", missing or "none")
save_table(dd, "data_dictionary")
dd

## WYI-9: Data-quality checks
Principle: **flag, don't silently delete.** Every issue gets a decision.

In [ ]:
# Cell 3 - Data-quality checks -> log table
num_cols = df.select_dtypes("number").columns.tolist()
df["process_date"] = pd.to_datetime(df["process_date"])
log = []
def add(check, result, decision):
    log.append({"check": check, "result": str(result), "decision": decision})

add("Row count", len(df), "Expected 1,250 rows (50 lots x 25 wafers)")
add("Missing values (total)", int(df.isna().sum().sum()), "None found" if df.isna().sum().sum()==0 else "Investigate")
add("Duplicate lot_id+wafer_id", int(df.duplicated(["lot_id","wafer_id"]).sum()), "Keep" if not df.duplicated(["lot_id","wafer_id"]).any() else "Review")
wpl = df.groupby("lot_id").size()
add("Wafers per lot (min/max)", f"{wpl.min()} / {wpl.max()}", "Consistent" if wpl.min()==wpl.max() else "Uneven - review")
add("Date range", f"{df.process_date.min().date()} to {df.process_date.max().date()}", "Only about 7 weeks - cannot study long-term drift")
add("Yield outside 0-1", int(((df['yield']<0)|(df['yield']>1)).sum()), "Valid" )
neg = {c:int((df[c]<0).sum()) for c in num_cols if (df[c]<0).any()}
add("Negative values in numeric columns", neg or "none", "OK" if not neg else "Review")

# Are tools / dates constant inside a lot? (matters for confounding)
for c in ["etch_tool","litho_tool","deposition_tool","implant_tool","product_type","technology_node","process_date"]:
    share = (df.groupby("lot_id")[c].nunique()==1).mean()
    add(f"{c} constant within lot", f"{share:.0%} of lots", "Lot-level attribute: all 25 wafers share it, so only 50 independent lots for this factor" if share==1 else "Varies within lot")

# Sanity: defect_count vs defect_density
add("corr(defect_count, defect_density)", round(df["defect_count"].corr(df["defect_density"]),3), "Near-duplicate information - note for modelling")
# Pattern to raise with client
add("Mean yield 7/10nm vs 14-28nm", f"{df[df.technology_node.isin(['7nm','10nm'])]['yield'].mean():.2f} vs {df[~df.technology_node.isin(['7nm','10nm'])]['yield'].mean():.2f}", "Unusual (advanced nodes usually yield lower) - ask client to confirm")

# IQR outliers (flag only)
out = {}
for c in num_cols:
    q1,q3 = df[c].quantile([.25,.75]); iqr=q3-q1
    out[c] = int(((df[c]<q1-1.5*iqr)|(df[c]>q3+1.5*iqr)).sum())
top = {k:v for k,v in sorted(out.items(), key=lambda x:-x[1]) if v>0}
add("IQR outliers per column (top)", dict(list(top.items())[:6]), "Flag only; confirm with client before any removal")
quality_log = pd.DataFrame(log)
save_table(quality_log, "data_quality_log")
quality_log

## WYI-10 and WYI-11: Yield KPI and clean table

In [ ]:
# Cell 4 - Define KPI, add flags, save clean (silver) table
LOW_YIELD_THRESHOLD = 0.30     # assumption - confirm with client (about 20% of wafers fall below)

clean = df.copy()
clean["low_yield_flag"] = (clean["yield"] < LOW_YIELD_THRESHOLD).astype(int)
flag_cols = []
for c in num_cols:
    if c == "yield": continue
    q1,q3 = clean[c].quantile([.25,.75]); iqr=q3-q1
    clean[f"out_{c}"] = ((clean[c]<q1-1.5*iqr)|(clean[c]>q3+1.5*iqr)).astype(int)
    flag_cols.append(f"out_{c}")
clean["outlier_count"] = clean[flag_cols].sum(axis=1)
clean = clean.drop(columns=flag_cols)             # keep one summary flag, keep table simple
clean["process_date"] = clean["process_date"].dt.date

print("Low-yield wafers:", int(clean.low_yield_flag.sum()), f"({clean.low_yield_flag.mean():.1%})")
print("Mean yield:", round(clean['yield'].mean(),3))
save_table(clean, "wafer_clean")

In [ ]:
# Cell 5 - Summary to paste into your Jira comments
print("ROWS:", len(clean), "| MISSING:", int(df.isna().sum().sum()), "| DUPLICATES:", int(df.duplicated(['lot_id','wafer_id']).sum()))
print("LOW-YIELD RATE:", f"{clean.low_yield_flag.mean():.1%}", "| MEAN YIELD:", round(clean['yield'].mean(),3))
quality_log[["check","result"]]